# Disease feature export (Colab Pro)

Scaffold for the **future** ML model. It does NOT train anything yet — there are no
ground-truth labels. It builds the labeled feature table that a model will eventually
be trained on, using the SAME satellites/indices/weather the production edge function
(`disease-risk-screen`) uses, so that what we train on matches what we serve.

Pipeline:
1. Authenticate Google Earth Engine (Python).
2. For each farm geometry, compute the disease indices (RBVI, CIre, MTCI, DWS,
   NDVI, NDVI_CV, **RIBInir, RIBIred, REDSI**) + the **thermal water-stress** proxy.
3. Join Open-Meteo weather (7-day lookback) at the farm centroid.
4. Join the **label** from `farmer_photo_submissions.diagnosis_result` (Stage-2 VLM /
   field confirmation) — the cell's confirmed disease, when available.
5. Export a feature CSV to Google Drive.

When enough labels accrue, a sibling notebook trains a logistic/RF model and exports
`disease_model.json` (schema = `DiseaseModelJSON` in `disease-models.ts`), which the
edge function loads via `mlDiseaseScore()` — a pure-TS, drop-in seam (no Python at
serve time).

**Column schema mirrors `disease_risk_cells`** so training features == serving features.

In [ ]:
# --- Setup ---
!pip -q install earthengine-api supabase requests pandas
import ee, requests, pandas as pd, datetime as dt
from supabase import create_client

# Auth GEE (uses the same service account as the edge function, or `ee.Authenticate()`)
ee.Authenticate()
ee.Initialize(project='YOUR_GEE_PROJECT')

# Supabase (read-only) — pull farm geometries + photo-confirmed labels
SUPABASE_URL = 'https://YOUR_PROJECT.supabase.co'
SUPABASE_KEY = 'YOUR_SERVICE_ROLE_OR_ANON_KEY'
sb = create_client(SUPABASE_URL, SUPABASE_KEY)

In [ ]:
# --- Disease indices (must stay in sync with optical-algorithms.ts) ---
S2_BANDS = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
S2_ALIAS = ['blue','green','red','rededge','rededge2','rededge3','nir','nir2','swir1','swir2']

def s2_composite(geom, start, end):
    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(geom).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
           .map(lambda im: im.select(S2_BANDS, S2_ALIAS).multiply(0.0001)))
    return col.median().clip(geom)

def disease_indices(c):
    B4, B5, B7, B8, B8A = (c.select('red'), c.select('rededge'), c.select('rededge3'),
                           c.select('nir'), c.select('nir2'))
    B11, B12 = c.select('swir1'), c.select('swir2')
    rbvi  = B8.multiply(9.78).subtract(B5.divide(B4).multiply(2.08)).clamp(-2,5).rename('rbvi')
    cire  = B8.divide(B5).subtract(1).clamp(0,10).rename('cire')
    mtci  = B8.subtract(B5).divide(B5.subtract(B4)).clamp(-5,10).rename('mtci')
    ndmi  = B8.subtract(B11).divide(B8.add(B11))
    nmdi  = B8.subtract(B11.subtract(B12)).divide(B8.add(B11.subtract(B12)))
    dws   = ndmi.multiply(0.6).add(nmdi.multiply(0.4)).clamp(-1,1).rename('dws')
    ndvi  = B8.subtract(B4).divide(B8.add(B4)).clamp(-1,1).rename('ndvi')
    # Published indices (same S2-band approximations as production)
    ribinir = B7.subtract(B8A).divide(B4.add(B8A)).clamp(-1,1).rename('ribinir')
    ribired = B5.subtract(B8A).divide(B4.add(B8A)).clamp(-1,1).rename('ribired')
    redsi   = (B7.subtract(B4).multiply(40)
               .subtract(B5.subtract(B4).multiply(118))
               .divide(B4.multiply(2).max(ee.Image(0.001))).clamp(-50,50).rename('redsi'))
    k = ee.Kernel.square(radius=3, units='pixels')
    mean = ndvi.reduceNeighborhood(ee.Reducer.mean(), k)
    sd   = ndvi.reduceNeighborhood(ee.Reducer.stdDev(), k)
    ndvi_cv = sd.divide(mean.abs().max(ee.Image(0.01))).clamp(0,2).rename('ndvi_cv')
    return ee.Image.cat([rbvi,cire,mtci,dws,ndvi,ndvi_cv,ribinir,ribired,redsi])

def thermal_stress(geom, start, end):
    t0 = (dt.date.fromisoformat(start) - dt.timedelta(days=21)).isoformat()
    def lst(im):
        return im.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST')
    ls = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
          .filterBounds(geom).filterDate(t0, end)
          .filter(ee.Filter.lt('CLOUD_COVER', 60)).map(lst))
    img = ls.median().select('LST')
    p = img.reduceRegion(ee.Reducer.percentile([5,95]), geom, 100, maxPixels=1e9, bestEffort=True)
    lo, hi = ee.Number(p.get('LST_p5')), ee.Number(p.get('LST_p95'))
    return img.subtract(ee.Image(lo)).divide(ee.Image(hi.subtract(lo).max(0.001))).clamp(0,1).rename('thermal_stress')

In [ ]:
# --- Open-Meteo weather (matches fetchWeatherRisk in the edge function) ---
def weather_features(lat, lng):
    url = ('https://api.open-meteo.com/v1/forecast'
           f'?latitude={lat}&longitude={lng}'
           '&hourly=temperature_2m,relative_humidity_2m,precipitation'
           '&past_days=7&forecast_days=1&timezone=Asia%2FKolkata')
    h = requests.get(url, timeout=30).json().get('hourly', {})
    t = h.get('temperature_2m', []); rh = h.get('relative_humidity_2m', []); rn = h.get('precipitation', [])
    return {
        'hours_temp_20_28c': sum(1 for x in t if 20 <= x <= 28),
        'leaf_wetness_hours': sum(1 for x in rh if x >= 80),
        'max_rh_pct': max(rh) if rh else 80,
        'total_rain_mm': sum(v or 0 for v in rn),
        'mean_temp_c': (sum(t)/len(t)) if t else 26,
    }

In [ ]:
# --- Build the labeled feature table ---
# Labels: photo-confirmed diagnosis per scout zone (centroid used as cell location).
subs = sb.table('farmer_photo_submissions').select(
    'farm_id, taken_lat, taken_lng, crop, growth_stage, diagnosis_result, submitted_at'
).not_.is_('diagnosis_result', 'null').execute().data

rows = []
for s in subs:
    diag = s.get('diagnosis_result') or {}
    label = diag.get('confirmed_diagnosis')
    if not label or label in ('unclear_image', 'uncertain'):
        continue
    lat, lng = s.get('taken_lat'), s.get('taken_lng')
    if lat is None or lng is None:
        continue
    end = (s.get('submitted_at') or dt.date.today().isoformat())[:10]
    start = (dt.date.fromisoformat(end) - dt.timedelta(days=14)).isoformat()
    pt = ee.Geometry.Point([lng, lat]).buffer(30)
    feats = disease_indices(s2_composite(pt, start, end)).addBands(thermal_stress(pt, start, end))
    vals = feats.reduceRegion(ee.Reducer.mean(), pt, 30, maxPixels=1e9, bestEffort=True).getInfo()
    vals.update(weather_features(lat, lng))
    vals['label'] = label
    vals['severity_pct'] = diag.get('severity_pct')
    vals['farm_id'] = s['farm_id']; vals['lat'] = lat; vals['lng'] = lng; vals['date'] = end
    rows.append(vals)

df = pd.DataFrame(rows)
print(f'{len(df)} labeled samples')
df.head()

In [ ]:
# --- Export to Drive ---
from google.colab import drive
drive.mount('/content/drive')
out = f"/content/drive/MyDrive/disease_features_{dt.date.today().isoformat()}.csv"
df.to_csv(out, index=False)
print('wrote', out)

# NEXT (separate notebook, once len(df) is large enough across diseases):
#   from sklearn.linear_model import LogisticRegression
#   train per-disease one-vs-rest logistic on the feature columns, then export
#   disease_model.json matching DiseaseModelJSON (features[], models{bias,coef[]}).
#   Drop that JSON into the edge function and call mlDiseaseScore().